In [1]:
import os
import pandas as pd
import numpy as np

from app_config import PROJ_ROOT, RAW_DATA_DIR_TRAIN, RAW_DATA_DIR_TEST
os.sys.path.append(os.path.join(PROJ_ROOT, "HumanActivityRecognition"))

from utils.transformations import *
from utils.transformations_utils import *

import os
import sys
import numpy as np
from app_config import PROJ_ROOT, RAW_DATA_DIR, RAW_DATA_DIR_TRAIN, RAW_DATA_DIR_TEST

sys.path.append(os.path.join(PROJ_ROOT, "HumanActivityRecognition"))

from utils import data_preprocessing
from utils.log_config import logger

from run_config import SLIDING_WINDOW_LENGTH
from run_config import NB_SENSOR_CHANNELS
from run_config import SLIDING_WINDOW_STEP

import sliding_window_on_data
from torch.utils.data import DataLoader, TensorDataset
from models.DeepConvLSTM import DeepConvLSTM, HARDataset, collate_fn,create_weighted_sampler

from utils.plot import plot_learning_curves
from utils import init_weights
import train

import optuna

2025-02-15 20:25:58.059 | INFO     | app_config:<module>:11 - PROJ_ROOT path is: C:\codes\HumanActivityRecognition
2025-02-15 20:26:04,770 - INFO - myapp - Logging configured from C:\codes\HumanActivityRecognition\HumanActivityRecognition\utils\base_config.json


No GPU available, training on CPU; consider making n_epochs very small.


2025-02-15 20:26:10,525 - DEBUG - matplotlib - matplotlib data path: c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\matplotlib\mpl-data
2025-02-15 20:26:10,593 - DEBUG - matplotlib - CONFIGDIR=C:\Users\carol\.matplotlib
2025-02-15 20:26:10,611 - DEBUG - matplotlib - interactive is False
2025-02-15 20:26:10,613 - DEBUG - matplotlib - platform is win32
2025-02-15 20:26:10,703 - DEBUG - matplotlib - CACHEDIR=C:\Users\carol\.matplotlib
2025-02-15 20:26:10,717 - DEBUG - matplotlib.font_manager - Using fontManager instance from C:\Users\carol\.matplotlib\fontlist-v390.json


No GPU available, training on CPU; consider making n_epochs very small.


c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Prepara i dati
datasetTracesTrain = data_preprocessing.build_dataset(RAW_DATA_DIR_TRAIN)
datasetTracesTest = data_preprocessing.build_dataset(RAW_DATA_DIR_TEST)
dataset_train_labled = data_preprocessing.add_labels_to_dataset(datasetTracesTrain)
dataset_test_labled = data_preprocessing.add_labels_to_dataset(datasetTracesTest)
X_Train, Y_Train = sliding_window_on_data.apply_sliding_window(dataset_train_labled, SLIDING_WINDOW_LENGTH, SLIDING_WINDOW_STEP, NB_SENSOR_CHANNELS)
np.save(os.path.join(RAW_DATA_DIR, 'X_Train.npy'), X_Train)

Loaded annotations and signals for TraceID: UMAH_User01_Activity01_Trial01
Annotations shape: (12024,)
Signals shape: (12024, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity01_Trial02
Annotations shape: (7214,)
Signals shape: (7214, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity01_Trial03
Annotations shape: (7114,)
Signals shape: (7114, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity02_Trial01
Annotations shape: (9819,)
Signals shape: (9819, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity02_Trial02
Annotations shape: (9518,)
Signals shape: (9518, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity02_Trial03
Annotations shape: (10120,)
Signals shape: (10120, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity03_Trial01
Annotations shape: (3707,)
Signals shape: (3707, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity03_Trial02
Annotations shape: (5210,

In [3]:
X_Train=np.load(os.path.join(RAW_DATA_DIR, 'X_Train.npy'))  
print(X_Train.shape)

(21942, 100, 9)


In [4]:
transform_funcs = [
    # transformations.scaling_transform_vectorized, # Use Scaling trasnformation
    noise_transform_vectorized, # Use rotation trasnformation
    scaling_transform_vectorized,
    #rotation_transform_vectorized,
    #axis_angle_to_rotation_matrix_3d_vectorized,
    negate_transform_vectorized,
    time_flip_transform_vectorized,
    channel_shuffle_transform_vectorized,
    #time_segment_permutation_transform_improved,
    #get_cubic_spline_interpolation,
    time_warp_transform_improved,
    time_warp_transform_low_cost,
]
transformation_function = generate_composite_transform_function_simple(transform_funcs)

0 <function noise_transform_vectorized at 0x000001A62E7A9120>
1 <function scaling_transform_vectorized at 0x000001A62E7CD000>
2 <function negate_transform_vectorized at 0x000001A63EFC1FC0>
3 <function time_flip_transform_vectorized at 0x000001A63EFC2050>
4 <function channel_shuffle_transform_vectorized at 0x000001A63EFC20E0>
5 <function time_warp_transform_improved at 0x000001A63EFC2290>
6 <function time_warp_transform_low_cost at 0x000001A63EFC2320>


In [5]:
# Apply transformation
# for i, riga in enumerate(X_train):
#     print(riga.shape)
#     print(riga)
#     print(transformation_function(riga))
#     transform_1 = transformation_function(riga)

tranform_1 = transformation_function(X_Train)

In [6]:
X_Train.shape, tranform_1.shape

#prima riga di X_train
X_Train[0]
print("Trasform")
#prima riga di X_train trasformata
tranform_1[0]



Trasform


array([[ 3.26843531e+00,  2.70839418e+01,  8.40421335e-01,
        -6.40981050e+01, -3.25590433e+01, -7.13852206e-02,
         1.95959179e+01,  8.65597031e+00, -5.44661820e-01],
       [ 3.27915649e+00,  2.70839418e+01,  8.40421335e-01,
        -6.40981050e+01, -3.25598802e+01, -7.13852206e-02,
         1.95959179e+01,  8.65597031e+00, -5.44661820e-01],
       [ 3.56406701e+00,  2.30161298e+01,  8.93682815e-01,
        -6.66638923e+01, -3.25733161e+01, -7.13852206e-02,
         1.97299291e+01,  9.66916574e+00, -5.45434197e-01],
       [ 3.68631182e+00,  1.43407600e+01,  8.08495657e-01,
        -7.14709704e+01, -3.21803614e+01, -1.12197684e-01,
         2.01644587e+01,  1.00723081e+01, -6.03102239e-01],
       [ 4.72131744e+00,  9.53293750e+00,  7.10777185e-01,
        -7.56559620e+01, -3.31056465e+01, -1.17571409e-01,
         2.09965240e+01,  1.04172002e+01, -6.50433983e-01],
       [ 6.01934892e+00,  7.32383124e+00,  7.86924855e-01,
        -7.81572082e+01, -3.21223540e+01, -1.386588

In [7]:
print(f"Dimensioni di X_Train: {X_Train.shape}")
print(f"Dimensioni di Y_Train: {Y_Train.shape}")
X_Train_augmented = np.concatenate((X_Train, tranform_1), axis=0)
Y_Train_augmented = np.concatenate((Y_Train, Y_Train), axis=0)
print(f"Dimensioni di X_Train_augmented: {X_Train_augmented.shape}")
print(f"Dimensioni di Y_Train_augmented: {Y_Train_augmented.shape}")

Dimensioni di X_Train: (21942, 100, 9)
Dimensioni di Y_Train: (21942,)
Dimensioni di X_Train_augmented: (43884, 100, 9)
Dimensioni di Y_Train_augmented: (43884,)


In [ ]:
train_dataset = HARDataset(X_Train_augmented, Y_Train)
test_dataset = HARDataset(X_Test, Y_Test)